In [ ]:
from google.colab import files
import os

print("Please upload your kaggle.json file:")
files.upload()

# Move the kaggle.json to the hidden folder
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download and unzip the dataset quietly
!kaggle datasets download -d techsash/waste-classification-data
!unzip -q waste-classification-data.zip
print("Dataset Downloaded and Extracted!")

In [ ]:
import tensorflow as tf
import numpy as np

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
DATA_DIR = "/content/DATASET/TRAIN"

print("Loading datasets...")
# 1. Load Data (shuffle=True is the default here, so it's already randomized safely!)
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="training", seed=123,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="validation", seed=123,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE)

class_names = train_ds.class_names

# 2. Optimize data loading (REMOVED .cache() and .shuffle() to fix the RAM crash)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

# 3. Native Data Augmentation (Handles varying backgrounds/rotations)
data_augmentation = tf.keras.Sequential([
  tf.keras.layers.RandomFlip("horizontal"),
  tf.keras.layers.RandomRotation(0.2),
  tf.keras.layers.RandomBrightness(factor=0.2),
])

# 4. MobileNetV2 Base Model
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')
base_model.trainable = False  # Freeze the base model

# 5. Build Our Custom Model
inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = preprocess_input(x) # Automatically scales pixels
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(len(class_names), activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

# 6. Compile and Train
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])

print("Training model...")
# 10 epochs
model.fit(train_ds, validation_data=val_ds, epochs=10)

# 7. Convert to standard TFLite
print("Converting to TFLite...")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open('model_unlabeled.tflite', 'wb') as f:
  f.write(tflite_model)
print("Plain TFLite model saved!")

In [ ]:
%%bash
echo "1. Building Python 3.10 Sandbox (takes about 30 seconds)..."
add-apt-repository ppa:deadsnakes/ppa -y &> /dev/null
apt-get update -q &> /dev/null
apt-get install python3.10 python3.10-distutils -y -q &> /dev/null
curl -sS https://bootstrap.pypa.io/get-pip.py | python3.10 &> /dev/null

echo "2. Installing the stable tflite-support library..."
python3.10 -m pip install tflite-support==0.4.4 -q

echo "3. Attaching Metadata..."
cat << 'EOF' > add_metadata.py
from tflite_support.metadata_writers import image_classifier
from tflite_support.metadata_writers import writer_utils

with open("labels.txt", "w") as f:
    f.write("Organic\nRecyclable\n")

writer = image_classifier.MetadataWriter.create_for_inference(
    writer_utils.load_file("model_unlabeled.tflite"),
    [0.0], [1.0], ["labels.txt"])

writer_utils.save_file(writer.populate(), "waste_classifier.tflite")
EOF

python3.10 add_metadata.py
echo "SUCCESS! Metadata is securely attached to waste_classifier.tflite"

In [ ]:
from google.colab import files
files.download('waste_classifier.tflite')